In [1]:
%%writefile main.py

# import torch
# import os
# from transformers import AutoModelForSequenceClassification, AutoConfig, AutoTokenizer, RobertaTokenizer
# import pandas as pd
# import sentencepiece

# def main():
#     checkpoint_path = '/kaggle/input/deberta-higher-lr-wd/transformers/default/1'  # Update to the provided checkpoint path

#     config = AutoConfig.from_pretrained(checkpoint_path)
#     model = AutoModelForSequenceClassification.from_config(config)

#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     print(device)
#     model = model.to(device)
#     model.eval()

#     test_df = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/test.csv')  # Update path if needed
#     test_df['combined_text'] = test_df['body'] + "[SEP]" + test_df['rule']
#     # Combining body and rule columns
#     tokenizer = AutoTokenizer.from_pretrained('/kaggle/input/huggingfacedebertav3variants/deberta-v3-base', use_fast=False)

#     batch_size = 8
#     predictions = []

#     # 6. Process data in smaller batches
#     for i in range(0, len(test_df), batch_size):
#         # Slice the test dataframe into smaller batches
#         batch = test_df.iloc[i:i + batch_size]
#         inputs = tokenizer(batch['combined_text'].tolist(), return_tensors="pt", padding="max_length", truncation='longest_first', max_length=128)
#         inputs = {key: val.to(device) for key, val in inputs.items()}

#         with torch.no_grad():
#             outputs = model(**inputs)
#             logits = outputs.logits
#             probs = torch.nn.functional.softmax(logits, dim=1)[:, 1].cpu().numpy()  # Get probabilities for the "rule_violation" class
#             predictions.extend(probs)

#         del inputs
#         torch.cuda.empty_cache()

#     submission_df = pd.DataFrame({
#         'row_id': test_df['row_id'],
#         'rule_violation': predictions
#     })

#     submission_df.to_csv('/kaggle/working/submission.csv', index=False)

# if __name__ == "__main__":
#     main()
import torch
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import numpy as np

# Function to load model and tokenizer, make predictions
def generate_predictions(model_path, tokenizer_path, checkpoint_path, test_df, batch_size=8, max_length=128):
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=False)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    
    predictions = []
    for i in range(0, len(test_df), batch_size):
        batch = test_df.iloc[i:i + batch_size]
        inputs = tokenizer(batch['combined_text'].tolist(), return_tensors="pt", padding="max_length", truncation='longest_first', max_length=max_length)
        inputs = {key: val.to(device) for key, val in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.nn.functional.softmax(logits, dim=1)[:, 1].cpu().numpy()
            predictions.extend(probs)
        
        del inputs
        torch.cuda.empty_cache()
    
    return predictions

# Read test data
test_df = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/test.csv')
test_df['combined_text'] = test_df['body'] + "[SEP]" + test_df['rule']  # Combining body and rule columns

# Define model paths and tokenizer paths
models = [
    ('/kaggle/input/deberta-v3-base-new/transformers/default/2', '/kaggle/input/huggingfacedebertav3variants/deberta-v3-base'),  # DeBERTa
    ('/kaggle/input/distilroberta-base-new/transformers/default/1', '/kaggle/input/d/sanjaypramod02/distilroberta-base'),  # DistilRoBERTa
    ('/kaggle/input/distilbert-base-uncased-new/transformers/default/1', '/kaggle/input/distillbert-uncased'),  # DistilBERT
]

# Generate and save predictions for each model
for model_path, tokenizer_path in models:
    predictions = generate_predictions(model_path, tokenizer_path, model_path, test_df)
    model_name = tokenizer_path.split('/')[-1]  # Using tokenizer path name as model name for saving
    submission_df = pd.DataFrame({'row_id': test_df['row_id'], 'rule_violation': predictions})
    submission_df.to_csv(f'/kaggle/working/submission_{model_name}.csv', index=False)


Writing main.py


In [2]:
!python main.py


2025-11-18 16:15:52.760567: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763482552.993272      39 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763482553.058523      39 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'


In [3]:
import pandas as pd
# Read the individual submission files
q = pd.read_csv('/kaggle/working/submission_deberta-v3-base.csv')  # DeBERTa submission (best model)
m = pd.read_csv('/kaggle/working/submission_distilroberta-base.csv')  # DistilRoBERTa submission
d = pd.read_csv('/kaggle/working/submission_distillbert-uncased.csv')  # DistilBERT submission

# Rank each model's predictions
rq = q['rule_violation'].rank(method='average') / (len(q) + 1)  # DeBERTa (best)
rm = m['rule_violation'].rank(method='average') / (len(m) + 1)
rd = d['rule_violation'].rank(method='average') / (len(d) + 1)

# Blend the ranks with custom weights (DeBERTa gets higher weight)
blend = 0.5 * rq + 0.3 * rm + 0.2 * rd  # DeBERTa is the best, so it gets the highest weight

# Assign the blended prediction back to the 'rule_violation' column
q['rule_violation'] = blend

# Save the final blended submission
q.to_csv('/kaggle/working/submission.csv', index=False)
